In [3]:
import pandas as pd

# 1. Load the data using your specific file path
input_path = 'C:/Users/vinay/OneDrive - Indian Institute of Technology Bhubaneswar/Desktop/sedrica assignment/city/crossing_evidence.csv'
df = pd.read_csv(input_path)

# 2. Compute Message Age
df['message_age'] = round(df['time_s'] - df['v2x_sample_time_s'], 2)

# 3. Define Thresholds
RED_THRESH = 0.5
PERSON_THRESH = 0.5
STALE_THRESH = 0.5 

# 4. Revised Decision Logic (Handles Stale Data)
def revised_decision(row):
    # Pedestrian safety first
    if row['person_score'] > PERSON_THRESH:
        return pd.Series(['STOP', 'Person detected'])
        
    v_red = row['red_score'] > RED_THRESH
    x_red = row['v2x_light'] == 'red'
    is_stale = row['message_age'] > STALE_THRESH
    
    # Traffic Light Logic
    if v_red: 
        if x_red:
            return pd.Series(['STOP', 'Agreement: Both say Red'])
        else:
            if is_stale:
                return pd.Series(['STOP', 'Camera Red, V2X Green but STALE. Trust Camera.'])
            else:
                return pd.Series(['SLOW', 'Unresolved Conflict: Fresh V2X Green vs Camera Red. Fallback SLOW.'])
    else: 
        if x_red:
            if is_stale:
                return pd.Series(['SLOW', 'Camera Green, V2X Red but STALE. Fallback SLOW.'])
            else:
                return pd.Series(['SLOW', 'Unresolved Conflict: Fresh V2X Red vs Camera Green. Fallback SLOW.'])
        else:
            if is_stale:
                return pd.Series(['SLOW', 'Agreement on Green, but V2X STALE. Fallback SLOW.'])
            else:
                return pd.Series(['GO', 'Agreement: Fresh V2X and Camera say Green'])

# 5. Apply the rule to create new columns
df[['revised_decision', 'revised_reason']] = df.apply(revised_decision, axis=1)

# 6. View a quick preview in the terminal
print(df[['episode', 'frame', 'v2x_light', 'message_age', 'revised_decision', 'revised_reason']].head(15))

# 7. Save the output permanently to your computer
output_path = 'C:/Users/vinay/OneDrive - Indian Institute of Technology Bhubaneswar/Desktop/sedrica assignment/city/revised_results.csv'
df.to_csv(output_path, index=False)

print(f"\nSuccess! Full revised results saved to: {output_path}")

        episode  frame v2x_light  message_age revised_decision  \
0   clear_green      0     green          0.0               GO   
1   clear_green      1     green          0.0               GO   
2   clear_green      2     green          0.0               GO   
3   clear_green      3     green          0.0               GO   
4   clear_green      4     green          0.0               GO   
5   clear_green      5     green          0.5               GO   
6   clear_green      6     green          0.6             STOP   
7   clear_green      7     green          0.0             STOP   
8   clear_green      8     green          0.0               GO   
9   clear_green      9     green          0.0             STOP   
10  clear_green     10     green          0.0               GO   
11  clear_green     11     green          0.0               GO   
12  clear_green     12     green          0.6             SLOW   
13  clear_green     13     green          0.6             SLOW   
14  clear_